In [1]:
!pip install langgraph langchain langchain-core PyMuPDF requests pandas numpy python-dotenv loguru pyyaml python-dateutil beautifulsoup4 cryptography aiohttp pytest sqlalchemy


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 152.4/152.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.5/216.5 kB 12.4 MB/s eta 0:00:00


In [2]:
from langgraph.graph import END, StateGraph
from langgraph.graph.graph import CompiledGraph
from typing import Dict, Any, List, Optional, TypedDict
import fitz  # PyMuPDF
import requests
import re
import json
from datetime import datetime, timedelta
import hashlib
import base64

# Define the state structure
class State(TypedDict):
    resume_text: str
    certifications: List[Dict[str, Any]]
    verification_results: List[List[Dict[str, Any]]]
    credibility_analysis: Dict[str, Any]
    interactive_mode: bool

# ---- Enhanced Certificate Extraction ----
def extract_certifications(text):
    """Extract certifications from resume text with improved parsing"""
    # This would normally use Gemini/LLM for better extraction
    # For now, using pattern matching and mock data

    # Pattern to find certification URLs
    url_pattern = r'https?://[^\s<>"{}|\\^`\[\]]+'
    urls = re.findall(url_pattern, text)

    # Mock extracted certifications with various verification methods
    certifications = [
        {
            "name": "Google TensorFlow Developer Certification",
            "issuer": "Google",
            "credential_id": "GTD-2024-001234",
            "api_endpoint": "https://api.google.com/credentials/verify",
            "verification_methods": ["api", "database", "issuer_website"]
        },
        {
            "name": "Coursera Python for Everybody",
            "issuer": "Coursera",
            "credential_id": "COURSERA-PY-567890",
            "link": "https://coursera.org/verify/ABC123DEF",
            "verification_methods": ["link", "api", "database"]
        },
        {
            "name": "AWS Cloud Practitioner Essentials",
            "issuer": "AWS",
            "credential_id": "AWS-CP-112233",
            "api_endpoint": "https://aws.amazon.com/verification/api",
            "verification_methods": ["api", "database"]
        },
        {
            "name": "Unstop AI Bootcamp",
            "issuer": "Unstop",
            "link": "https://unstop.com/certificate/ABC123" if urls else None,
            "verification_methods": ["link", "manual"]
        },
        {
            "name": "HackerRank Python Basic",
            "issuer": "HackerRank",
            "link": "https://www.hackerrank.com/certificates/XYZ456" if urls else None,
            "credential_id": "HR-PY-445566",
            "verification_methods": ["link", "api"]
        }
    ]

    return certifications

# ---- Enhanced Verification Database ----
fake_cert_db = {
    "Google TensorFlow Developer Certification": {
        "status": "valid",
        "platform": "Google",
        "issued_date": "2024-01-15",
        "expiry_date": "2026-01-15",
        "credential_hash": "a1b2c3d4e5f6"
    },
    "Coursera Python for Everybody": {
        "status": "valid",
        "platform": "Coursera",
        "issued_date": "2023-11-20",
        "expiry_date": None,  # No expiry
        "credential_hash": "f6e5d4c3b2a1"
    },
    "AWS Cloud Practitioner Essentials": {
        "status": "expired",
        "platform": "AWS",
        "issued_date": "2022-06-10",
        "expiry_date": "2024-06-10",
        "credential_hash": "123abc456def"
    },
    "Unstop AI Bootcamp": {
        "status": "not_found",
        "platform": "Unstop"
    },
    "HackerRank Python Basic": {
        "status": "valid",
        "platform": "HackerRank",
        "issued_date": "2024-03-01",
        "expiry_date": None,
        "credential_hash": "def456abc123"
    },
}

# ---- Multiple Verification Methods ----
def verify_by_api(cert: Dict[str, Any]) -> Dict[str, Any]:
    """Verify certificate using API endpoint"""
    api_endpoint = cert.get("api_endpoint")
    credential_id = cert.get("credential_id")

    if not api_endpoint or not credential_id:
        return {"method": "api", "status": "insufficient_data", "message": "Missing API endpoint or credential ID"}

    try:
        # Mock API call
        payload = {"credential_id": credential_id, "issuer": cert["issuer"]}
        # In real implementation: response = requests.post(api_endpoint, json=payload, timeout=10)

        # Simulate API response based on our mock database
        cert_name = cert["name"]
        if cert_name in fake_cert_db:
            db_entry = fake_cert_db[cert_name]
            if db_entry["status"] == "valid":
                return {
                    "method": "api",
                    "status": "verified",
                    "message": f"Certificate verified via API",
                    "details": {
                        "issued_date": db_entry.get("issued_date"),
                        "expiry_date": db_entry.get("expiry_date"),
                        "credential_hash": db_entry.get("credential_hash")
                    }
                }
            elif db_entry["status"] == "expired":
                return {
                    "method": "api",
                    "status": "expired",
                    "message": f"Certificate expired on {db_entry.get('expiry_date')}",
                    "details": db_entry
                }

        return {"method": "api", "status": "not_found", "message": "Certificate not found in issuer database"}

    except Exception as e:
        return {"method": "api", "status": "error", "message": f"API verification failed: {str(e)}"}

def verify_by_link(cert: Dict[str, Any]) -> Dict[str, Any]:
    """Verify certificate using direct link"""
    link = cert.get("link")

    if not link:
        return {"method": "link", "status": "no_link", "message": "No verification link provided"}

    try:
        # Mock link verification
        response = requests.get(link, timeout=10, allow_redirects=True)

        if response.status_code == 200:
            # Check if the page contains certificate validation indicators
            content = response.text.lower()
            cert_indicators = ["certificate", "credential", "verified", "valid", "authentic"]

            if any(indicator in content for indicator in cert_indicators):
                return {
                    "method": "link",
                    "status": "verified",
                    "message": f"Certificate verified via link: {link}",
                    "details": {"url": link, "response_code": response.status_code}
                }
            else:
                return {
                    "method": "link",
                    "status": "suspicious",
                    "message": "Link accessible but certificate validity unclear"
                }
        else:
            return {
                "method": "link",
                "status": "invalid_link",
                "message": f"Link returned status {response.status_code}"
            }

    except requests.exceptions.Timeout:
        return {"method": "link", "status": "timeout", "message": "Link verification timed out"}
    except Exception as e:
        return {"method": "link", "status": "error", "message": f"Link verification failed: {str(e)}"}

def verify_by_database(cert: Dict[str, Any]) -> Dict[str, Any]:
    """Verify certificate against internal database"""
    cert_name = cert["name"]

    if cert_name in fake_cert_db:
        db_entry = fake_cert_db[cert_name]
        return {
            "method": "database",
            "status": db_entry["status"],
            "message": f"Database lookup: {db_entry['status']}",
            "details": db_entry
        }

    return {
        "method": "database",
        "status": "not_found",
        "message": "Certificate not found in database"
    }

def verify_by_issuer_website(cert: Dict[str, Any]) -> Dict[str, Any]:
    """Verify certificate by checking issuer's official website"""
    issuer = cert["issuer"]
    credential_id = cert.get("credential_id")

    # Mock issuer website verification
    issuer_endpoints = {
        "Google": "https://cloud.google.com/certification/verify",
        "AWS": "https://aws.amazon.com/verification",
        "Microsoft": "https://learn.microsoft.com/en-us/certifications/verify",
        "Coursera": "https://coursera.org/verify"
    }

    if issuer in issuer_endpoints:
        # Simulate website verification
        return {
            "method": "issuer_website",
            "status": "verified" if cert["name"] in fake_cert_db and fake_cert_db[cert["name"]]["status"] == "valid" else "not_found",
            "message": f"Checked on {issuer} official website",
            "details": {"verification_url": issuer_endpoints[issuer]}
        }

    return {
        "method": "issuer_website",
        "status": "unsupported",
        "message": f"No verification endpoint available for {issuer}"
    }

def manual_verification_prompt(cert: Dict[str, Any]) -> Dict[str, Any]:
    """Prompt for manual verification when automatic methods fail"""
    return {
        "method": "manual",
        "status": "requires_manual_review",
        "message": f"Manual verification required for {cert['name']}",
        "details": {
            "instructions": "Please provide additional verification documents or contact the issuer directly",
            "issuer_contact": f"Contact {cert['issuer']} for verification"
        }
    }

# ---- Enhanced Verification Logic ----
def verify_certificate_comprehensive(cert: Dict[str, Any]) -> List[Dict[str, Any]]:
    """Comprehensive verification using multiple methods"""
    verification_methods = cert.get("verification_methods", ["database"])
    results = []

    for method in verification_methods:
        if method == "api":
            results.append(verify_by_api(cert))
        elif method == "link":
            results.append(verify_by_link(cert))
        elif method == "database":
            results.append(verify_by_database(cert))
        elif method == "issuer_website":
            results.append(verify_by_issuer_website(cert))
        elif method == "manual":
            results.append(manual_verification_prompt(cert))

    return results

# ---- Enhanced Scoring Logic ----
def calculate_credibility_score(all_results: List[List[Dict[str, Any]]]) -> Dict[str, Any]:
    """Calculate comprehensive credibility score"""
    total_score = 0
    max_possible_score = 0
    detailed_breakdown = []

    for cert_results in all_results:
        cert_score = 0
        cert_max = 100
        verification_count = len(cert_results)

        # Weight verification methods
        method_weights = {
            "api": 40,
            "issuer_website": 35,
            "database": 25,
            "link": 20,
            "manual": 10
        }

        for result in cert_results:
            method = result["method"]
            status = result["status"]
            weight = method_weights.get(method, 10)

            if status == "verified":
                cert_score += weight
            elif status == "expired":
                cert_score += weight * 0.3  # Partial credit for expired but authentic
            elif status == "suspicious":
                cert_score += weight * 0.1
            # No points for not_found, error, etc.

        # Bonus for multiple verification methods
        if verification_count > 1:
            cert_score = min(cert_score * 1.1, cert_max)

        total_score += cert_score
        max_possible_score += cert_max

        detailed_breakdown.append({
            "certificate": cert_results[0].get("certificate_name", "Unknown"),
            "score": cert_score,
            "max_score": cert_max,
            "verification_methods": len(cert_results),
            "results": cert_results
        })

    overall_score = (total_score / max_possible_score * 100) if max_possible_score > 0 else 0

    return {
        "overall_score": round(overall_score, 2),
        "total_certificates": len(all_results),
        "detailed_breakdown": detailed_breakdown,
        "scoring_criteria": {
            "api_verification": "40 points",
            "issuer_website": "35 points",
            "database_lookup": "25 points",
            "link_verification": "20 points",
            "manual_review": "10 points",
            "multiple_methods_bonus": "10% bonus"
        }
    }

# ---- User Interaction for Credential URLs ----
def prompt_for_credential_urls(certifications: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Prompt user for additional credential URLs if available"""
    updated_certs = []

    for cert in certifications:
        print(f"\n📜 Found certification: {cert['name']}")
        print(f"   Issuer: {cert['issuer']}")

        if not cert.get("link") and not cert.get("api_endpoint"):
            response = input(f"   Do you have a verification URL for this certificate? (y/n): ").lower()
            if response == 'y':
                url = input(f"   Please enter the verification URL: ").strip()
                if url:
                    cert["link"] = url
                    if "link" not in cert.get("verification_methods", []):
                        cert["verification_methods"] = cert.get("verification_methods", []) + ["link"]

        if not cert.get("credential_id"):
            response = input(f"   Do you have a credential ID for this certificate? (y/n): ").lower()
            if response == 'y':
                cred_id = input(f"   Please enter the credential ID: ").strip()
                if cred_id:
                    cert["credential_id"] = cred_id

        updated_certs.append(cert)

    return updated_certs

# ---- Enhanced LangGraph Nodes ----
def extract_node(state: State) -> State:
    """Extract certifications from resume text"""
    text = state["resume_text"]
    certs = extract_certifications(text)
    return {"certifications": certs}

def user_input_node(state: State) -> State:
    """Allow user to provide additional credential information"""
    certs = state["certifications"]

    # Check if interactive mode is enabled
    if state.get("interactive_mode", False):
        updated_certs = prompt_for_credential_urls(certs)
        return {"certifications": updated_certs}

    return {"certifications": certs}

def verify_node(state: State) -> State:
    """Perform comprehensive verification of all certificates"""
    certs = state["certifications"]
    all_results = []

    for cert in certs:
        verification_results = verify_certificate_comprehensive(cert)
        # Add certificate name to results for tracking
        for result in verification_results:
            result["certificate_name"] = cert["name"]
        all_results.append(verification_results)

    return {"verification_results": all_results}

def score_node(state: State) -> State:
    """Calculate comprehensive credibility score"""
    results = state["verification_results"]
    score_data = calculate_credibility_score(results)
    return {"credibility_analysis": score_data}

def flagging_feedback_node(state: State) -> State:
    """Analyze verification results and flag issues with suggestions"""
    flagged = []
    results = state["verification_results"]

    for cert_results in results:
        cert_name = cert_results[0].get("certificate_name", "Unknown")
        for result in cert_results:
            if result["status"] in ["expired", "not_found", "invalid_link", "suspicious", "error", "timeout"]:
                flagged.append({
                    "certificate": cert_name,
                    "method": result["method"],
                    "issue": result["status"],
                    "message": result["message"],
                    "suggestion": suggest_remediation(result["status"])
                })

    return {"credibility_analysis": {**state["credibility_analysis"], "flags": flagged}}

def suggest_remediation(status: str) -> str:
    """Provide suggestions based on the status"""
    suggestions = {
        "expired": "Renew the certification if possible.",
        "not_found": "Check the certification name or contact the issuer.",
        "invalid_link": "Update with a correct or working verification link.",
        "suspicious": "Provide additional proof or check legitimacy.",
        "error": "Retry later or contact issuer support.",
        "timeout": "Check network or try a different method."
    }
    return suggestions.get(status, "Manual review recommended.")


# ---- Build Enhanced LangGraph ----
builder = StateGraph(State)

builder.add_node("extract", extract_node)
builder.add_node("user_input", user_input_node)
builder.add_node("verify", verify_node)
builder.add_node("score", score_node)
builder.add_node("flag_feedback", flagging_feedback_node)

builder.set_entry_point("extract")
builder.add_edge("extract", "user_input")
builder.add_edge("user_input", "verify")
builder.add_edge("verify", "score")
builder.add_edge("score", "flag_feedback")
builder.add_edge("flag_feedback", END)

graph: CompiledGraph = builder.compile()

# ---- Enhanced Usage Function ----
def run_comprehensive_verification(text: str, interactive: bool = False) -> Dict[str, Any]:
    """Run comprehensive certificate verification"""
    initial_state = {
        "resume_text": text,
        "interactive_mode": interactive,
        "certifications": [], # Initialize with empty list
        "verification_results": [], # Initialize with empty list
        "credibility_analysis": {} # Initialize with empty dict
    }
    return graph.invoke(initial_state)

# ---- Pretty Print Results ----
def print_verification_report(result: Dict[str, Any]):
    """Print a comprehensive verification report"""
    print("\n" + "="*80)
    print("🏆 CERTIFICATE VERIFICATION REPORT")
    print("="*80)

    # Overall Score
    analysis = result.get("credibility_analysis", {})
    overall_score = analysis.get("overall_score", 0)

    print(f"\n📊 OVERALL CREDIBILITY SCORE: {overall_score}/100")
    print(f"📜 Total Certificates Analyzed: {analysis.get('total_certificates', 0)}")

    # Detailed Breakdown
    print(f"\n🔍 DETAILED VERIFICATION RESULTS:")
    print("-" * 50)

    for breakdown in analysis.get("detailed_breakdown", []):
        cert_name = breakdown["certificate"]
        cert_score = breakdown["score"]
        max_score = breakdown["max_score"]

        print(f"\n📋 {cert_name}")
        print(f"   Score: {cert_score:.1f}/{max_score}")
        print(f"   Verification Methods Used: {breakdown['verification_methods']}")

        for result in breakdown["results"]:
            method = result["method"].title()
            status = result["status"].title().replace("_", " ")
            message = result["message"]
            print(f"   • {method}: {status} - {message}")

    # Scoring Criteria
    print(f"\n📏 SCORING CRITERIA:")
    print("-" * 30)
    criteria = analysis.get("scoring_criteria", {})
    for criterion, points in criteria.items():
        print(f"   • {criterion.replace('_', ' ').title()}: {points}")

    flags = analysis.get("flags", [])
    if flags:
        print(f"\n🚩 FLAGGED ISSUES & FEEDBACK:")
        print("-" * 40)
        for flag in flags:
            print(f"\n❗ Certificate: {flag['certificate']}")
            print(f"   Method: {flag['method']}")
            print(f"   Issue: {flag['issue']}")
            print(f"   Message: {flag['message']}")
            print(f"   Suggested Action: {flag['suggestion']}")
    else:
        print("\n✅ No major issues flagged. All certifications passed initial checks.")

# ---- Example Usage ----
if __name__ == "__main__":
    sample_text = """
        Sridhar S has the following certifications:
        - Google TensorFlow Developer Certification (ID: GTD-2024-001234)
        - Coursera Python for Everybody
        - AWS Cloud Practitioner Essentials
        - Unstop AI Bootcamp: https://unstop.com/certificate/ABC123
        - HackerRank Python Basic: https://www.hackerrank.com/certificates/XYZ456
    """

    print("🚀 Starting Comprehensive Certificate Verification...")

    # Run with interactive mode disabled for demo
    result = run_comprehensive_verification(sample_text, interactive=False)

    # Print comprehensive report
    print_verification_report(result)

    # For interactive mode, uncomment the line below:
    # result = run_comprehensive_verification(sample_text, interactive=True)

🚀 Starting Comprehensive Certificate Verification...

🏆 CERTIFICATE VERIFICATION REPORT

📊 OVERALL CREDIBILITY SCORE: 25.19/100
📜 Total Certificates Analyzed: 5

🔍 DETAILED VERIFICATION RESULTS:
--------------------------------------------------

📋 Google TensorFlow Developer Certification
   Score: 82.5/100
   Verification Methods Used: 3
   • Api: Verified - Certificate verified via API
   • Database: Valid - Database lookup: valid
   • Issuer_Website: Verified - Checked on Google official website

📋 Coursera Python for Everybody
   Score: 22.0/100
   Verification Methods Used: 3
   • Link: Verified - Certificate verified via link: https://coursera.org/verify/ABC123DEF
   • Api: Insufficient Data - Missing API endpoint or credential ID
   • Database: Valid - Database lookup: valid

📋 AWS Cloud Practitioner Essentials
   Score: 21.5/100
   Verification Methods Used: 2
   • Api: Expired - Certificate expired on 2024-06-10
   • Database: Expired - Database lookup: expired

📋 Unstop AI B